# 06 - Playground SQL

Este notebook fica como espaco livre para testar consultas novas sobre o `olist_colab.sqlite`.

In [ ]:
from pathlib import Path
import sqlite3
import urllib.request
import zipfile

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)

PROJECT_DIR = Path.cwd()
DB_ZIP_URL = "https://github.com/Urpia-S/Olist_E-commerce_Analytic-SQL-Python/releases/download/data-v1/olist_colab.sqlite.zip"
DB_PATH = PROJECT_DIR / "olist_colab.sqlite"


def baixar_banco_da_release():
    zip_path = PROJECT_DIR / "olist_colab.sqlite.zip"
    urllib.request.urlretrieve(DB_ZIP_URL, zip_path)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(PROJECT_DIR)

    print("Banco extraido em:", DB_PATH)


if not DB_PATH.exists():
    baixar_banco_da_release()

conn = sqlite3.connect(DB_PATH)


def consulta(sql):
    return pd.read_sql_query(sql, conn)

## Tabelas e views disponiveis

Lista rapida para lembrar os objetos criados no banco.

In [ ]:
consulta("""
-- Objetos disponiveis no SQLite.
SELECT
    type AS tipo,
    name AS nome
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name;
""")

## Exemplo 1: cidades com mais pedidos

In [ ]:
cidades = consulta("""
-- Cidades com maior quantidade de pedidos.
SELECT
    c.customer_city,
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS pedidos
FROM core_orders o
JOIN core_customers c ON c.customer_id = o.customer_id
GROUP BY c.customer_city, c.customer_state
ORDER BY pedidos DESC
LIMIT 15;
""")

cidades

## Exemplo 2: comentarios negativos

In [ ]:
consulta("""
-- Exemplos de comentarios negativos para leitura qualitativa.
SELECT
    review_score,
    review_comment_title,
    review_comment_message
FROM core_order_reviews
WHERE review_score IN (1, 2)
  AND review_comment_message IS NOT NULL
LIMIT 20;
""")

## Exemplo 3: media movel mensal de receita

In [ ]:
media_movel = consulta("""
-- Receita mensal com media movel de 3 meses.
WITH mensal AS (
    SELECT
        strftime('%Y-%m', o.order_purchase_timestamp) AS mes_pedido,
        SUM(oi.price) AS receita_produtos
    FROM core_orders o
    JOIN core_order_items oi ON oi.order_id = o.order_id
    WHERE o.order_purchase_timestamp IS NOT NULL
    GROUP BY strftime('%Y-%m', o.order_purchase_timestamp)
)
SELECT
    mes_pedido,
    ROUND(receita_produtos, 2) AS receita_produtos,
    ROUND(AVG(receita_produtos) OVER (
        ORDER BY mes_pedido
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS media_movel_3_meses
FROM mensal
ORDER BY mes_pedido;
""")

ax = media_movel.plot(x="mes_pedido", y=["receita_produtos", "media_movel_3_meses"], figsize=(11, 4))
ax.set_title("Receita mensal e media movel de 3 meses")
ax.set_xlabel("")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

media_movel.tail()

## Area livre

Uso este bloco para testar outras consultas.

In [ ]:
minha_consulta = """
-- Consulta livre para exploracao.
SELECT *
FROM vw_order_items_enriched
LIMIT 10;
"""

consulta(minha_consulta)